# Unit 7 exercises: plan the lap, then find its edge

A planned speed profile brakes before the corner instead of in it, and a few knobs decide how hard it pushes.
These exercises build the profile from its two passes, follow it with a lead, test the lesson's claims on the oval, look at the friction circle along the grid's edge, and tune knobs with the clean rule.
Every number here is a number about racekit's **carla-like car**: a model, not the truth.

Run the setup cell first.
Each exercise names the variables your answer must define; fill in every `None` and `"TODO"`.
Functions you write in one exercise are reused in later ones, so run the exercises in order.
Every race uses `carla_like(noise=True)` with the stated seed; a race takes a few seconds, and the exercises keep the number of races small.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import grip_sections, load_track

AGENTS = Path("../../projects/racekit/agents")
UNIT_06, UNIT_07 = AGENTS / "unit-06.py", AGENTS / "unit-07.py"
workdir = Path(mkdtemp(prefix="roar-unit-07-ex-"))
CLEAN_OFFSET = 2.0  # m: the clean rule's limit on the max |cross-track error|
CONTROL_DT = 0.05  # s: one telemetry row per control tick


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, written into workdir.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def max_offset(result):
    # The largest |cross-track error| over the driving rows (held == 0).
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def clean(result):
    # The unit's clean rule: finished, 0 collisions, and max |cross-track error| at most 2 m.
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET

## Exercise 1 - The two passes on a straight

A toy track, not a loop: 601 waypoints numbered 0 … 600 along a straight line, every segment 1 m long (`ds[i] = 1.0` is the segment from waypoint i to i + 1, so `ds` has 600 entries).
The cap is 45 m/s everywhere except a corner at waypoints 300 … 320 (both included), where it is 15 m/s.

Write two functions that each return a **new list** and do not go round a loop (no wrap-around):

- `backward_pass(cap, ds, a_dec)`: start from a copy of `cap`; for i from the second-to-last index **down** to 0, set v[i] = min(v[i], √(v[i + 1]² + 2·a_dec·ds[i]));
- `forward_pass(v, ds, a_acc)`: start from a copy of `v`; for i from 0 **up** to the second-to-last index, set v[i + 1] = min(v[i + 1], √(v[i]² + 2·a_acc·ds[i])).

Then compute the toy profile, `forward_pass(backward_pass(cap, ds, 5.0), ds, 4.0)`.

Define:

- `backward_pass`, `forward_pass`: the functions;
- `toy_cap`: the cap, a list of 601 floats;
- `toy_profile`: the profile, a list of 601 floats;
- `ramp_start`: the smallest index i for which `toy_profile[i + 1] < toy_profile[i] - 1e-9` (where the braking ramp begins);
- `ramp_formula`: the braking distance from 45 m/s to 15 m/s at 5 m/s², (45² − 15²) / (2·5), in metres;
- `back_to_top`: the smallest index i > 320 for which `toy_profile[i] >= 45.0 - 1e-9`;
- `planned_time`: the time to drive the profile, the sum over the 600 segments of ds[i] / ((v[i] + v[i + 1]) / 2), in seconds;
- `cap_time`: the same sum for `toy_cap` itself;
- `why_not_the_cap`: one sentence on why `cap_time` is not a time any car could drive.

In [ ]:
def backward_pass(cap, ds, a_dec):
    return None  # TODO: a new list


def forward_pass(v, ds, a_acc):
    return None  # TODO: a new list


toy_cap = []  # TODO
toy_profile = []  # TODO
ramp_start = None  # TODO
ramp_formula = None  # TODO
back_to_top = None  # TODO
planned_time = None  # TODO
cap_time = None  # TODO
why_not_the_cap = "TODO"

## Exercise 2 - The whole lap, on the oval

Write `lap_profile(xs, ys, a_lat, a_dec, a_acc, top_speed=45.0, laps=2)` for a closed lap of n waypoints (indices wrap round: waypoint n is waypoint 0):

- κ_i is the Menger curvature of waypoints (i − 1, i, i + 1), cyclic, as in Unit 2 (only |κ_i| matters here);
- ds_i is the straight-line length of the segment from waypoint i to waypoint i + 1 (the last segment closes the lap);
- the cap is min(top_speed, √(a_lat / |κ_i|)) where |κ_i| > 1e-9, and top_speed elsewhere;
- the backward pass: `laps` times, for i from n − 1 **down** to 0, v_i = min(v_i, √(v_{(i+1) mod n}² + 2·a_dec·ds_i));
- then the forward pass: `laps` times, for i from 0 **up** to n − 1, v_{(i+1) mod n} = min(v_{(i+1) mod n}, √(v_i² + 2·a_acc·ds_i)).

It returns the profile as a NumPy array of n floats.

Use the `oval` (`track = load_track("oval")`, waypoints `track.x`, `track.y`) with the reference's numbers, a_lat = 4.6, a_dec = 5.0, a_acc = 4.0.
Compare with the profile the submission plans: load its class and build an agent from the waypoints alone,

```python
with shim_installed():
    Unit07, _ = load_submission(UNIT_07)
agent = Unit07(track.waypoints, None)  # agent.profile is its list of planned speeds
```

Define:

- `lap_profile`: the function;
- `oval_profile`: its result on the oval with `laps=2`;
- `largest_difference`: the largest |oval_profile − agent.profile| over the waypoints (m/s);
- `changed_by_second_lap`: the number of waypoints where the profiles with `laps=1` and `laps=2` differ by more than 1e-9;
- `slowest_index`, `slowest_speed`: the waypoint of the lowest planned speed (the first one if there is a tie) and that speed;
- `planned_lap_time`: the sum over the n segments (cyclic) of ds_i / ((v_i + v_{(i+1) mod n}) / 2), in seconds;
- `fraction_at_cap`: the fraction of waypoints where the profile equals the cap to within 1e-9;
- `fig`, `ax`: one plot of the cap (clipped at 45) and the profile against the waypoint index, with axis labels, a legend and a title.

In [ ]:
def lap_profile(xs, ys, a_lat, a_dec, a_acc, top_speed=45.0, laps=2):
    return None  # TODO: a NumPy array of n speeds


oval_profile = None  # TODO
largest_difference = None  # TODO
changed_by_second_lap = None  # TODO
slowest_index, slowest_speed = None, None  # TODO
planned_lap_time = None  # TODO
fraction_at_cap = None  # TODO
fig, ax = None, None  # TODO

## Exercise 3 - A lead in seconds

The submission reads its speed goal a **lead** ahead of the car: starting at the car's waypoint `current_index`, it walks forward, adding up segment lengths, until the path covered reaches max(`PROFILE_LEAD_S`·v, `LEAD_MIN_M`); the goal is the profile at the waypoint reached.
Exactly:

```python
j, ahead = current_index, 0.0
while ahead < lead:
    ahead += segment_length[j % n]
    j += 1
# the goal is profile[j % n]
```

Write `lead_index(segment_length, current_index, speed, lead_s=0.3, lead_min=2.0)` that returns `j % n`.

Use `test-circuit`, and build an agent from its waypoints as in Exercise 2; `agent.segment_length` is its list of segment lengths and `agent.profile` its planned speeds.
Put the car on waypoint 0 and use the speeds 0, 5, 10, 20, 30 and 45 m/s.
(Waypoint 0 of `test-circuit` is on the way out of the last corner, so the profile there is still below 45 m/s.)

Define:

- `lead_index`: the function;
- `lead_metres`: a dict from speed to max(0.3·v, 2.0) (m);
- `lead_waypoints`: a dict from speed to `lead_index(...)` for the car on waypoint 0;
- `goal_speed`: a dict from speed to the profile at that index (m/s);
- `two_waypoint_lead_s`: the time a car at 45 m/s takes to cover the first two segments of `test-circuit` (`segment_length[0] + segment_length[1]`), in seconds;
- `why_seconds`: one or two sentences: the speed loop needs time to brake to the profile; why is a lead counted in seconds a better rule than a lead counted in waypoints, at different speeds and on a track like CARLA's?

In [ ]:
def lead_index(segment_length, current_index, speed, lead_s=0.3, lead_min=2.0):
    return None  # TODO: j % n


lead_metres = {}  # TODO
lead_waypoints = {}  # TODO
goal_speed = {}  # TODO
two_waypoint_lead_s = None  # TODO
why_seconds = "TODO"

## Exercise 4 - The friction circle along the edge

The lesson's (A_LAT, A_DEC) grid on `test-circuit` has a clean edge in every row.
The last clean A_LAT of each A_DEC row is:

| A_DEC (m/s²) | 3.0 | 4.0 | 4.5 | 5.0 | 6.0 |
|---|---|---|---|---|---|
| last clean A_LAT (m/s²) | 5.2 | 5.0 | 4.8 | 4.8 | 4.4 |

If the edge were set by the tyres' friction circle alone, a_lat² + a_dec² ≤ (μg)², the planned rates at the edge would sit on a circle: a_lat² + a_dec² would be the same in every row.

Define:

- `edge`: a dict from A_DEC to the last clean A_LAT, as in the table;
- `circle_sums`: a dict from A_DEC to a_lat² + a_dec² at the edge (m²/s⁴);
- `combined`: a dict from A_DEC to √(a_lat² + a_dec²) (m/s²);
- `fraction_of_g`: a dict from A_DEC to `combined[a_dec] / 9.81`;
- `sum_range`: the largest minus the smallest of `circle_sums`;
- `is_constant`: `True` when `sum_range` is below 1.0;
- `fig`, `ax`: the five edge points in the (A_DEC, A_LAT) plane, with the quarter circle of radius `combined[3.0]` (A_DEC from 0 to that radius) drawn for comparison, equal axis scales, axis labels and a title;
- `why_not_constant`: two or three sentences: the car obeys a friction circle, so why do the planned rates at the edge not lie on one? (Think about what the planner models and what it does not: its two passes, the speed loop that follows the profile, and the line pure pursuit drives.)

In [ ]:
edge = {}  # TODO
circle_sums = {}  # TODO
combined = {}  # TODO
fraction_of_g = {}  # TODO
sum_range = None  # TODO
is_constant = None  # TODO
fig, ax = None, None  # TODO
why_not_constant = "TODO"

## Exercise 5 - The gain over Unit 6, on the oval

Race `agents/unit-06.py` and `agents/unit-07.py`, each with `repeat_races` on `"oval"`, seeds 0, 1 and 2 (`n=3, seeds=(0, 1, 2)`), `car_factory=lambda: carla_like(noise=True)` and `max_seconds=600.0`.

Define:

- `unit_6_results`, `unit_7_results`: the two `RepeatResult` objects;
- `unit_6_times`, `unit_7_times`: lists of the three race times (s), in seed order;
- `all_clean`: `True` when all six races are clean by the setup cell's `clean`;
- `medians`, `spreads`: dicts `{"unit-06": ..., "unit-07": ...}` of the median race time and the spread (max − min) (s);
- `unit_7_better`: `noise_aware_better(unit_7_times, unit_6_times)`;
- `gain_s`: the Unit 6 median minus the Unit 7 median (s);
- `gain_fraction`: `gain_s` divided by the Unit 6 median;
- `three_seeds`: one sentence: is three seeds enough for **this** comparison, and would it be enough to compare two Unit 7 settings 0.3 s apart?

In [ ]:
unit_6_results, unit_7_results = None, None  # TODO
unit_6_times, unit_7_times = [], []  # TODO
all_clean = None  # TODO
medians, spreads = {}, {}  # TODO
unit_7_better = None  # TODO
gain_s = None  # TODO
gain_fraction = None  # TODO
three_seeds = "TODO"

## Exercise 6 - Where the time goes

Use the **seed-0** race of each agent from Exercise 5 (`unit_6_results.results[0]` and `unit_7_results.results[0]`), and the oval's grip sections, `sections = grip_sections(load_track("oval"))`.

racekit's telemetry `progress` counts from the waypoint **after** the start: a row with progress p is at waypoint (p + 1) mod n, where n is the number of waypoints in one lap.
So a row's section is `sections[(p + 1) % n]`.
A section's time is the number of rows with `held == 0` in that section, times `CONTROL_DT` (summed over the three laps).

Define:

- `section_of_rows(telemetry, sections)`: a function returning a NumPy array with the section of every row;
- `times_6`, `times_7`: lists of the six section times (s) for Unit 6 and Unit 7;
- `section_gain`: a list of the six differences `times_6[k] - times_7[k]` (s);
- `biggest_gain_section`, `smallest_gain_section`: the sections with the largest and the smallest gain (gains are whole numbers of ticks, so ties are possible: if two sections tie, take the lower section number);
- `peak_speed_7`: a list of Unit 7's highest speed in each section (m/s, rows with `held == 0`; the speed loop can overshoot `TARGET_SPEED` a little);
- `fig`, `ax`: one plot of `times_6` and `times_7` against the section number (`ax.plot` with markers, one line per agent), with a legend, axis labels and a title;
- `explain`: one or two sentences: why does Unit 7 gain more in some sections than in others? (Look at the track: where are the straights?)

In [ ]:
def section_of_rows(telemetry, sections):
    return None  # TODO


times_6, times_7 = [], []  # TODO
section_gain = []  # TODO
biggest_gain_section, smallest_gain_section = None, None  # TODO
peak_speed_7 = []  # TODO
fig, ax = None, None  # TODO
explain = "TODO"

## Exercise 7 - The edge on one row

The reference plans with A_DEC = 5.0 and A_LAT = 4.6; the lesson's grid says the edge of that row is at A_LAT 4.8.
Check it on one seed.
Race variants of `agents/unit-07.py` with `A_LAT` = 4.8 and with `A_LAT` = 5.0 (A_DEC stays 5.0) on `"test-circuit"`, `car=carla_like(noise=True)`, `seed=0` and `max_seconds=400.0`.

Define:

- `row`: a dict from A_LAT (4.8 and 5.0) to a tuple (clean, race time in s, collisions, max |cross-track error| in m);
- `edge_on_seed_0`: the larger of the two A_LAT values whose race is clean (`None` if neither is);
- `margin_choice`: the A_LAT the margin rule ships for this row, one grid step of 0.2 below `edge_on_seed_0` (rounded to 1 decimal);
- `seconds_given_up`: the race time at `margin_choice` minus the race time at `edge_on_seed_0`, on seed 0, using 304.15 s for A_LAT 4.6 (the lesson's seed-0 race of the reference) if `margin_choice` is 4.6;
- `one_seed_caveat`: one sentence: what would you have to race before believing `edge_on_seed_0` is the edge, by the unit's rule?

In [ ]:
row = {}  # TODO
edge_on_seed_0 = None  # TODO
margin_choice = None  # TODO
seconds_given_up = None  # TODO
one_seed_caveat = "TODO"

## Exercise 8 - Challenge: a backward pass that respects the friction circle

The profile's backward pass brakes at the full A_DEC even where the car is cornering.
A tyre cannot do both at full strength.
Make the braking pass share the grip, on an ellipse: on the segment from waypoint i to i + 1, the braking available is

$$a_i = A_{\text{DEC}}\,\sqrt{\max\!\left(0,\ 1 - \left(\frac{v_{i+1}^2\,|\kappa_{i+1}|}{A_{\text{LAT}}}\right)^2\right)},$$

with v_{i+1} the profile's current speed at waypoint i + 1 (already lowered by the pass) and κ_{i+1} its Menger curvature.
So a waypoint cornering at its full cap allows no braking on the segment that leads into it, and a straight allows the full A_DEC.

Write `circle_profile(cap, curvature, segment_length, a_lat, a_dec, a_acc, laps=2)`: start from a copy of `cap`; the backward pass `laps` times round the loop (i from n − 1 down to 0, j = (i + 1) mod n), v_i = min(v_i, √(v_j² + 2·a_i·ds_i)) with the a_i above; then the **unchanged** forward pass `laps` times (i from 0 up to n − 1, v_j = min(v_j, √(v_i² + 2·a_acc·ds_i))).
Return a list.

Use `test-circuit` and an agent built from its waypoints as in Exercise 2: `agent.cap`, `agent.curvature` and `agent.segment_length` are the inputs, with a_lat = 4.6, a_dec = 5.0, a_acc = 4.0; `agent.profile` is the plain profile.
Then race it: subclass the loaded class so that its `__init__` calls the parent's and then replaces `self.profile` with `circle_profile(self.cap, self.curvature, self.segment_length, 4.6, 5.0, 4.0)`, and pass the subclass itself to `run_race` on `"test-circuit"` with `car=carla_like(noise=True)`, `seed=0` and `max_seconds=600.0`.

Define:

- `circle_profile`: the function;
- `circle`: its result on `test-circuit` (a list);
- `lowered_count`: the number of waypoints where `circle` is lower than `agent.profile` by more than 1e-9;
- `largest_drop`: the largest `agent.profile[i] - circle[i]` (m/s);
- `planned_lap_plain`, `planned_lap_circle`: the planned lap times, the sum over the n segments (cyclic) of ds_i / ((v_i + v_{(i+1) mod n}) / 2) (s);
- `circle_race`: a tuple (clean, race time in s, max |cross-track error| in m) of the race;
- `worth_it`: two sentences: compare with the reference's seed-0 race (clean, 304.15 s, max |offset| 1.48 m); what did respecting the circle buy and cost, and which knob would you move next to win the time back?

In [ ]:
def circle_profile(cap, curvature, segment_length, a_lat, a_dec, a_acc, laps=2):
    return None  # TODO: a list


circle = None  # TODO
lowered_count = None  # TODO
largest_drop = None  # TODO
planned_lap_plain, planned_lap_circle = None, None  # TODO
circle_race = None  # TODO
worth_it = "TODO"

## Exercise 9 - Challenge: does the order of the sequential pass matter?

The lesson's sequential pass swept the sections in the order 0, 1, 2, 3, 4, 5.
Run it in both orders, on a cheap setting: `test-circuit`, **one lap** (`laps=1`), `car=carla_like(noise=True)`, `seed=0`, `max_seconds=200.0`, the clean rule of the setup cell.

The pass: start from `GRIP = (1.0,) * 6`.
For each section in the order, try the knobs 1.1, 1.2, 1.3, 1.4, 1.5 in turn: set that section's knob, keeping every other section's current knob, race a variant of `agents/unit-07.py` with that `GRIP` (a tuple of six floats), and stop that section at its first race that is not clean.
The section keeps the last clean knob (1.0 if 1.1 was not clean), and the pass moves on to the next section on top of it.
One lap and one seed is a cheap setting, so the knobs it finds need not match the lesson's five-seed, three-lap pass; compare the two orders with each other, not with the lesson.

Define:

- `sequential_pass(order)`: a function that returns (the final GRIP tuple, the number of races it ran);
- `forward_knobs`, `forward_races`: the result for the order 0 → 5;
- `backward_knobs`, `backward_races`: the result for the order 5 → 0;
- `same_knobs`: `True` when the two final tuples are equal;
- `why_order_matters`: two or three sentences: explain the result in terms of what one section's knob does to the next section's entry speed, and say what this means for a per-section tuning found by any one order.

In [ ]:
def sequential_pass(order):
    return None, 0  # TODO: (GRIP tuple, number of races)


forward_knobs, forward_races = None, None  # TODO
backward_knobs, backward_races = None, None  # TODO
same_knobs = None  # TODO
why_order_matters = "TODO"